# 06 · Comparison arm (a): pure embedding search (CLIP-style)

The A1 design, kept as a comparison: a shared space trained on automatic captions (`configs/captions.yaml`) links
the question text (frozen `bge-small` sentence encoder plus a projection) with each pixel's AlphaEarth embedding
(land head) and its POWER cell's summary-statistics vector (climate head). Training is three-way and
multi-positive (a caption's positives are all batch items it is true for). Used alone: the question's text
embedding ranks every valid pixel of the pool's test tiles. Scored by average precision (AP) and by F1 of the
top-k pixels, with k = the number of true pixels (the arm is given the true answer size).

Checks (fixed in `PLAN.md`): **planted** land vectors (class, distance and patch facts written out exactly) and
**shuffled captions**.

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import average_precision_score

from askmap.captions import Location, Vocab, fact_text, make_caption
from askmap.layers import class_mask
from askmap.local_facts import distances, leaf_patch_ha, patch_area_ha, sample_pixels
from askmap.score import median_ci
from askmap.space import Items, Specs, TrainData, search, train_space

CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
SM, MC = cfg["sampling"], cfg["model"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
f = np.load(ROOT / "climate" / "facts.npz")
FACT_IDS, TABLE = [str(x) for x in f["ids"]], f["table"]
FAMILY = np.array(["long" if i.split("|")[0] == "sun" else i.split("|")[0] for i in FACT_IDS])
SUMMARY = np.load(ROOT / "climate" / "enc_summary.npy")
vocab = Vocab(CAP, LEAVES, GROUPS, FACT_IDS)
NEAR = vocab.near_classes
OTHER = LEAVES.index("other")
RES_M = cfg["grid"]["res_m"]


def local(leaf):
    return {
        "dist": distances(leaf, NEAR, LEAVES, GROUPS, RES_M),
        "patch_leaf": leaf_patch_ha(leaf, len(LEAVES), RES_M),
        "patch_crop": patch_area_ha(class_mask(leaf, "cropland", LEAVES, GROUPS), RES_M),
    }


def planted(leaf, lf, flat):
    d = lf["dist"].reshape(-1, len(NEAR))[flat]
    parts = [np.eye(len(LEAVES), dtype=np.float32)[leaf.ravel()[flat]]]
    parts += [
        (d[:, [k]] <= np.array(CAP["near"]["distances_m"])[None, :]).astype(np.float32)
        for k in range(len(NEAR))
    ]
    for key in ("patch_leaf", "patch_crop"):
        a = lf[key].ravel()[flat]
        parts.append((a[:, None] >= np.array(CAP["patch"]["sizes_ha"])[None, :]).astype(np.float32))
    return np.concatenate(parts, axis=1)

/Volumes/disco2/github/ask-the-map-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1030 12:53:35.660000 7014 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Training locations (training tiles of every area) and captions

In [3]:
parts = []
for k, (area, acfg) in enumerate(cfg["areas"].items()):
    for j, (name, (tile, role)) in enumerate(area_tiles(acfg, cfg["grid"]).items()):
        if role != "train":
            continue
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        leaf = official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar)
        valid = np.isfinite(z["ae"]).all(axis=0) & (leaf != OTHER)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        cell = uni.index(box, iy, ix)
        lf = local(leaf)
        flat = sample_pixels(leaf, valid, SM["train_pixels_per_tile"], len(LEAVES), SM["seed"] + 100 * k + j)
        parts.append(
            {
                "ae": z["ae"].reshape(64, -1)[:, flat].T.astype(np.float32),
                "leaf": leaf.ravel()[flat],
                "dist": lf["dist"].reshape(-1, len(NEAR))[flat],
                "patch_leaf": lf["patch_leaf"].ravel()[flat],
                "patch_crop": lf["patch_crop"].ravel()[flat],
                "cell": cell.ravel()[flat],
                "planted": planted(leaf, lf, flat),
                "area": np.full(len(flat), area),
            }
        )
LOC = {key: np.concatenate([p[key] for p in parts]) for key in parts[0]}
assert not cells.heldout.to_numpy()[LOC["cell"]].any()
rng = np.random.default_rng(SM["seed"])
cache = {}


def true_facts(c):
    if c not in cache:
        on = np.flatnonzero(TABLE[c])
        cache[c] = {fam: on[FAMILY[on] == fam] for fam in ("hist", "heat", "rain", "gdd", "long")}
    return cache[c]


rows = []
for j in range(len(LOC["leaf"])):
    loc = Location(
        int(LOC["leaf"][j]),
        LOC["dist"][j],
        float(LOC["patch_leaf"][j]),
        float(LOC["patch_crop"][j]),
        true_facts(int(LOC["cell"][j])),
        str(LOC["area"][j]),
    )
    for _ in range(SM["captions_per_location"]):
        c = make_caption(loc, vocab, rng)
        rows.append(
            {
                "item": j,
                "full": c["full"],
                "land_text": c["land"],
                "clim_text": c["clim"],
                **{
                    "land_mask": c["spec"]["land"],
                    "near_mask": c["spec"]["near"],
                    "type": c["spec"]["type"],
                    "d": c["spec"]["d"],
                    "neg": c["spec"]["neg"],
                    "patch_kind": c["spec"]["patch_kind"],
                    "h": c["spec"]["h"],
                    "fact": c["spec"]["fact"],
                },
            }
        )
caps = pd.DataFrame(rows)
comp = CAP["composition"]
fams = ["hist", "heat", "rain", "gdd", "long"]
fam_p = np.array([comp["climate_year"] / 4] * 4 + [comp["climate_long"]])
co_rows = []
for c in np.flatnonzero(~cells.heldout.to_numpy()):
    tf = true_facts(int(c))
    for _ in range(SM["climate_only_captions_per_cell"]):
        order = [fm for fm in rng.choice(fams, size=5, replace=False, p=fam_p / fam_p.sum()) if len(tf[fm])]
        fid = int(rng.choice(tf[order[0]]))
        co_rows.append(
            {
                "cell": int(c),
                "fact": fid,
                "text": str(rng.choice(CAP["prefixes"]))
                + fact_text(FACT_IDS[fid], CAP, int(rng.integers(3))),
            }
        )
co = pd.DataFrame(co_rows)
print(f"{len(LOC['leaf']):,} locations, {len(caps):,} captions, {len(co):,} climate-only captions")
caps.full.sample(10, random_state=0).tolist()

85,000 locations, 510,000 captions, 21,750 climate-only captions


['Find fallow or idle cropland in the driest third of places by May-September root-zone soil wetness over 2001-2025',
 'Places with farmland under annual crops',
 'built-up land at least 100 m away from a lake or pond',
 'roads, farmyards and scattered houses',
 'Show wetlands',
 'Places with cropland in the lowest half of the region for yearly cooling degree days, 2001-2025',
 'grassland and pasture where May-September 2013 had at most 1300 growing degree days (base 10 C)',
 'fields of sunflowers',
 'Places with soybean fields',
 'Find open water']

## Pools (all valid pixels of each pool's test tiles) and text embeddings

In [4]:
POOLS = {}
for pool in POOL_Q:
    ae, cl, pl = [], [], []
    for n in TILES[pool]["test"]:
        flat = np.flatnonzero(T[n]["valid"].ravel())
        z = np.load(ROOT / "tiles" / f"{T[n]['area']}_{n}.npz")["ae"]
        ae.append(z.reshape(64, -1)[:, flat].T.astype(np.float32))
        cl.append(T[n]["cell"].ravel()[flat])
        pl.append(planted(T[n]["official"], local(T[n]["official"]), flat))
    POOLS[pool] = {"ae": np.concatenate(ae), "cell": np.concatenate(cl), "planted": np.concatenate(pl)}
TRUTH = dict(np.load(ROOT / "maps" / "truth.npz"))
QROWS = []
for q in Q:
    seen = q["plan"]["area"]
    for wording, pool in (("as_written", seen), ("reworded", seen), ("as_written", HELD[seen])):
        n = len(POOLS[pool]["cell"])
        g = np.unpackbits(TRUTH[f"{q['id']}|{pool}"])[:n].astype(bool)
        QROWS.append(
            {
                "id": q["id"],
                "wording": wording,
                "pool": pool,
                "text": question_text(q, wording, pool),
                "truth": g,
                "persona": q["persona"],
                "n_extra": n_extra(q["plan"]),
                "has_climate": bool(q["plan"]["climate"]),
            }
        )
strings = pd.unique(
    pd.concat(
        [
            caps.full,
            caps.land_text,
            caps.clim_text[caps.clim_text != ""],
            co.text,
            pd.Series([r["text"] for r in QROWS]),
        ]
    ).to_numpy()
)
t0 = time.time()
enc = SentenceTransformer(cfg["text_encoder"], device=str(device))
E = enc.encode(
    list(strings), batch_size=512, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
).astype(np.float32)
pos = {s: i for i, s in enumerate(strings)}
print(
    f"{len(strings):,} texts encoded in {time.time() - t0:.0f}s; pools",
    {p: len(v["cell"]) for p, v in POOLS.items()},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7124.65it/s]

324,889 texts encoded in 144s; pools {'north_dakota': 1334667, 'north_carolina': 1334667, 'minnesota': 1334647, 'georgia': 1334667}


## Train and search

In [5]:
def tens(a, dtype=torch.float32):
    return torch.as_tensor(np.asarray(a), dtype=dtype, device=device)


items = Items(
    tens(LOC["leaf"], torch.long),
    tens(LOC["dist"]),
    tens(LOC["patch_leaf"]),
    tens(LOC["patch_crop"]),
    tens(LOC["cell"], torch.long),
)
specs = Specs(
    tens(np.stack(caps.land_mask.to_numpy()), torch.bool),
    tens(caps.type, torch.long),
    tens(np.stack(caps.near_mask.to_numpy()), torch.bool),
    tens(caps.d),
    tens(caps.neg, torch.bool),
    tens(caps.patch_kind, torch.long),
    tens(caps.h),
    tens(caps.fact, torch.long),
)
cell_facts = tens(TABLE, torch.bool)
tf, tl = caps.full.map(pos).to_numpy(), caps.land_text.map(pos).to_numpy()
tc = caps.clim_text.map(lambda s: pos.get(s, -1)).to_numpy()
ct = co.text.map(pos).to_numpy()


def run(name, land_key, shuffle=False):
    t0 = time.time()
    a, b, c, d = tf, tl, tc, ct
    if shuffle:
        r = np.random.default_rng(MC["seed"])
        perm = r.permutation(len(caps))
        a, b, c, d = a[perm], b[perm], c[perm], d[r.permutation(len(co))]
    data = TrainData(
        items=items,
        land=tens(LOC[land_key]),
        clim=tens(SUMMARY),
        cap_item=tens(caps.item, torch.long),
        t_full=tens(E[a]),
        t_land=tens(E[b]),
        t_clim=tens(E[np.where(c >= 0, c, 0)]),
        has_clim=tens(c >= 0, torch.bool),
        specs=specs,
        co_cell=tens(co.cell, torch.long),
        co_text=tens(E[d]),
        co_fact=tens(co.fact, torch.long),
        cell_facts=cell_facts,
    )
    model, hist = train_space(data, MC, False, device, log=lambda *_: None)
    out = []
    for pool, pdata in POOLS.items():
        rows_p = [r for r in QROWS if r["pool"] == pool]
        sims = search(
            model, E[[pos[r["text"]] for r in rows_p]], pdata[land_key], SUMMARY[pdata["cell"]], device
        )
        for r, s in zip(rows_p, sims, strict=True):
            g, k = r["truth"], int(r["truth"].sum())
            top = np.argpartition(-s, k)[:k] if k else np.array([], dtype=int)
            out.append(
                {
                    "model": name,
                    "id": r["id"],
                    "wording": r["wording"],
                    "pool": pool,
                    "heldout": pool not in SEEN,
                    "persona": r["persona"],
                    "n_extra": r["n_extra"],
                    "has_climate": r["has_climate"],
                    "truth_px": k,
                    "prevalence": float(g.mean()),
                    "AP": average_precision_score(g, s) if k else np.nan,
                    "F1_topk": float(g[top].mean()) if k else np.nan,
                }
            )
    print(f"{name}: {time.time() - t0:.0f}s, loss {hist[0]:.3f} -> {hist[-1]:.3f}")
    return pd.DataFrame(out)


res = pd.concat([run("clip", "ae"), run("clip_planted", "planted"), run("clip_shuffled", "ae", shuffle=True)])
res["scored"] = res.truth_px >= SC["min_truth"]
res.to_csv(RES / "06_embedding_search.csv", index=False)

/var/folders/9b/g2cxr52j3gz6bg6tpjjscd5w0000gn/T/ipykernel_7014/2711540077.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  return torch.as_tensor(np.asarray(a), dtype=dtype, device=device)


clip: 766s, loss 0.230 -> 0.131


clip_planted: 739s, loss 0.097 -> 0.030


clip_shuffled: 758s, loss 1.569 -> 1.533


## Checks and summary

In [6]:
B, S = SC["bootstrap"], SC["seed"]


def s1(df):
    return df[df.scored & (df.wording == "as_written") & ~df.heldout & (df.n_extra <= 1)]


pl = median_ci(s1(res[res.model == "clip_planted"]).AP.to_numpy(), B, S)
sh = s1(res[res.model == "clip_shuffled"])
shl = median_ci((sh.AP - sh.prevalence).to_numpy(), B, S)
V = {
    "planted_S1_median_AP": pl,
    "planted_pass": pl[0] >= 0.80,
    "shuffled_S1_median_lift": shl,
    "shuffled_pass": shl[0] <= 0.02,
    "minutes": round((time.time() - T_START) / 60, 1),
}
(RES / "06_embedding_search.json").write_text(json.dumps(V, indent=1, default=float))
print(V)
res[res.scored].groupby(["model", "wording", "heldout"])[["AP", "F1_topk", "prevalence"]].median().round(3)

{'planted_S1_median_AP': (0.5416906410978279, 0.4724480885303663, 0.8251818520370231), 'planted_pass': False, 'shuffled_S1_median_lift': (-0.0011734121172898397, -0.005408939834939695, -3.139125450205525e-05), 'shuffled_pass': True, 'minutes': 43.2}


AP  F1_topk  prevalence
model         wording    heldout                            
clip          as_written False    0.387    0.422       0.056
                         True     0.099    0.106       0.026
              reworded   False    0.364    0.371       0.056
clip_planted  as_written False    0.533    0.532       0.056
                         True     0.503    0.451       0.026
              reworded   False    0.502    0.514       0.056
clip_shuffled as_written False    0.068    0.020       0.056
                         True     0.041    0.000       0.026
              reworded   False    0.066    0.031       0.056